# 03 - Linear Regression\n
\n
Baseline em Spark MLlib com split temporal, encoding de categoricas e persistencia de metricas.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/notebooks')

import time

import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression

from _lib import (
    build_spark, append_metric,
    SEED, SPLIT_DATE, SILVER_PATH, TARGET_COL,
    NUMERIC_COLS, CATEGORICAL_COLS,
)

MODEL_OUTPUT = '/models/linear_regression'
RESIDUAL_PLOT = '/results/residual_analysis_linear_regression.png'

spark = build_spark('nyc-rideshare-linear-regression')
sns.set_theme(style='whitegrid')

In [ ]:
silver_df = spark.read.parquet(SILVER_PATH)
silver_df.createOrReplaceTempView('trips_silver')

# Split temporal via Spark SQL puro (regra do projeto: data prep em SQL, ML pipeline em Python).
train_df = spark.sql(f"""
    SELECT * FROM trips_silver
    WHERE pickup_datetime < TIMESTAMP '{SPLIT_DATE}'
""").cache()
test_df = spark.sql(f"""
    SELECT * FROM trips_silver
    WHERE pickup_datetime >= TIMESTAMP '{SPLIT_DATE}'
""").cache()

train_rows = train_df.count()
test_rows = test_df.count()
if train_rows == 0 or test_rows == 0:
    raise ValueError('O split temporal exige dados antes e depois de 2023-06-01. Um unico mes como 2023-08 nao basta para treinar e avaliar.')

train_rows, test_rows

In [ ]:
indexers = [\n
    StringIndexer(inputCol=col_name, outputCol=f'{col_name}_idx', handleInvalid='keep')\n
    for col_name in CATEGORICAL_COLS\n
]\n
encoders = [\n
    OneHotEncoder(inputCol=f'{col_name}_idx', outputCol=f'{col_name}_ohe')\n
    for col_name in CATEGORICAL_COLS\n
]\n
numeric_assembler = VectorAssembler(inputCols=NUMERIC_COLS, outputCol='numeric_features')\n
scaler = StandardScaler(inputCol='numeric_features', outputCol='scaled_numeric_features', withMean=False, withStd=True)\n
feature_assembler = VectorAssembler(\n
    inputCols=['scaled_numeric_features'] + [f'{col_name}_ohe' for col_name in CATEGORICAL_COLS],\n
    outputCol='features'\n
)\n
lr = LinearRegression(\n
    labelCol=TARGET_COL,\n
    featuresCol='features',\n
    predictionCol='prediction',\n
    regParam=0.0,\n
    elasticNetParam=0.0\n
)\n
\n
pipeline = Pipeline(stages=indexers + encoders + [numeric_assembler, scaler, feature_assembler, lr])

In [ ]:
start_time = time.perf_counter()\n
lr_model = pipeline.fit(train_df)\n
train_seconds = time.perf_counter() - start_time\n
\n
lr_model.write().overwrite().save(MODEL_OUTPUT)\n
print(f'Modelo salvo em {MODEL_OUTPUT} | treino: {train_seconds:.2f}s')

In [ ]:
predictions = lr_model.transform(test_df).cache()\n
evaluators = {\n
    'rmse': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='rmse'),\n
    'mae': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='mae'),\n
    'r2': RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='r2')\n
}\n
metrics = {name: evaluator.evaluate(predictions) for name, evaluator in evaluators.items()}\n
metrics

In [ ]:
predictions.createOrReplaceTempView('predictions_lr')\n
\n
spark.sql("""\n
SELECT\n
    pu_borough,\n
    ROUND(SQRT(AVG(POWER(base_passenger_fare - prediction, 2))), 4) AS rmse,\n
    ROUND(AVG(ABS(base_passenger_fare - prediction)), 4) AS mae,\n
    COUNT(*) AS rows\n
FROM predictions_lr\n
GROUP BY 1\n
ORDER BY rmse DESC\n
""").show(truncate=False)\n
\n
spark.sql("""\n
SELECT\n
    CASE\n
        WHEN trip_miles < 2 THEN '0-2 mi'\n
        WHEN trip_miles < 5 THEN '2-5 mi'\n
        WHEN trip_miles < 10 THEN '5-10 mi'\n
        ELSE '10+ mi'\n
    END AS distance_bucket,\n
    ROUND(SQRT(AVG(POWER(base_passenger_fare - prediction, 2))), 4) AS rmse,\n
    ROUND(AVG(ABS(base_passenger_fare - prediction)), 4) AS mae,\n
    COUNT(*) AS rows\n
FROM predictions_lr\n
GROUP BY 1\n
ORDER BY distance_bucket\n
""").show(truncate=False)

In [ ]:
residual_pdf = (predictions\n
    .select('prediction', TARGET_COL)\n
    .sample(False, 0.02, SEED)\n
    .toPandas())\n
residual_pdf['residual'] = residual_pdf[TARGET_COL] - residual_pdf['prediction']\n
\n
fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n
sns.histplot(residual_pdf['residual'], bins=50, ax=axes[0])\n
axes[0].set_title('Distribuicao dos residuos - LR')\n
sns.scatterplot(data=residual_pdf, x='prediction', y='residual', s=10, alpha=0.3, ax=axes[1])\n
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)\n
axes[1].set_title('Residuo vs predito - LR')\n
plt.tight_layout()\n
plt.savefig(RESIDUAL_PLOT, dpi=150, bbox_inches='tight')\n
plt.show()

In [ ]:
results_df = append_metric(
    model='linear_regression',
    rmse=metrics['rmse'],
    mae=metrics['mae'],
    r2=metrics['r2'],
    train_seconds=train_seconds,
    notes='baseline com StandardScaler e one-hot encoding',
)
results_df.tail(10)

In [ ]:
spark.stop()